In [ ]:
from pyspark.sql.functions import current_timestamp

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working. Schema/volume/storage are the same across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
catalog = dbutils.widgets.get("catalog")

table_name = "vline_trip_updates"
table_path = f"{catalog}.`01_bronze`.{table_name}"
base       = "abfss://lakehouse@transportvicdatalake.dfs.core.windows.net/landing/vline_trip_updates/"
checkpoint = f"/Volumes/{catalog}/01_bronze/_checkpoints/{table_name}"

bronze = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "binaryFile")
      .option("cloudFiles.schemaLocation", checkpoint + "/schema")
      .load(base)
      .withColumn("_ingest_ts", current_timestamp())
)

(bronze.writeStream
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable(table_path)
 )